# Island annotation overlap (mm39)

How many projected query islands overlap existing GENCODE / tRNA annotations in mm39?
This gives the **"Y% of matched islands overlap existing ncRNA annotations"** number for the manuscript.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

REPO_ROOT = Path("..")
sys.path.insert(0, str(REPO_ROOT / "modules"))

RESULTS_DIR = REPO_ROOT / "preprint_results" / "hg38_vs_mm39"
ANNOTATION_DIR = REPO_ROOT / "input_data" / "mm39_annotation_validation"
FIGURES_DIR = Path("raw_plots")
FIGURES_DIR.mkdir(exist_ok=True)

QUERY_ISLANDS_BED = RESULTS_DIR / "query_annotation" / "raw_query_islands.bed"
ANNO_BED = ANNOTATION_DIR / "mm39_gencode_all_transcripts.bed"
TRNA_BED = ANNOTATION_DIR / "mm39-tRNAs.bed"
META_TSV = RESULTS_DIR / "reference_union_transcripts_metadata.tsv"

print(f"Query islands BED:  {QUERY_ISLANDS_BED}")
print(f"GENCODE annotation: {ANNO_BED}")
print(f"tRNA annotation:    {TRNA_BED}")
print(f"Metadata TSV:       {META_TSV}")

## Load query islands and annotation

In [ ]:
from pyrion.io.bed import read_narrow_bed_file, read_bed12_file
from pyrion.ops.interval_ops import intersect_intervals

islands = read_bed12_file(str(QUERY_ISLANDS_BED))
annotation = read_bed12_file(str(ANNO_BED))
trna_annotation = read_bed12_file(str(TRNA_BED))

print(f"Query islands:       {len(islands):>9,}")
print(f"GENCODE transcripts: {len(annotation):>9,}")
print(f"tRNA genes:          {len(trna_annotation):>9,}")

In [ ]:
meta = pd.read_csv(META_TSV, sep="\t")
bt_map = meta.set_index("transcript_id")["biotype"].to_dict()

def parse_island_name(name):
    """'U_ENSG00000294950.1.1_island_0' -> transcript_id='U_ENSG00000294950.1', island_idx=0"""
    parts = name.rsplit("_island_", 1)
    island_idx = int(parts[1]) if len(parts) == 2 else -1
    # transcript_id is everything before the last dot-number (chain_id)
    prefix = parts[0]
    dot_parts = prefix.split(".")
    transcript_id = ".".join(dot_parts[:-1])  # drop chain_id suffix
    return transcript_id, island_idx

# Quick test
print(parse_island_name("U_ENSG00000294950.1.1_island_0"))
print(parse_island_name("U_ENSG00000284738.3.10001_island_2"))

## Intersect each query island with mm39 annotations

In [ ]:
rows = []
n = len(islands)
progress_step = max(1, n // 20)

for i, isl in enumerate(islands):
    if i % progress_step == 0:
        print(f"  {i:,}/{n:,} ...")

    isl_arr = np.array([[isl.start, isl.end]])
    isl_len = isl.end - isl.start
    isl_name = isl.id or f"{isl.chrom}:{isl.start}-{isl.end}"
    tx_id, isl_idx = parse_island_name(isl_name)
    biotype = bt_map.get(tx_id, "unknown")

    # GENCODE overlap
    best_bp = 0
    best_tid = None
    for t in annotation.get_transcripts_in_interval(isl):
        isect = intersect_intervals(isl_arr, t.blocks)
        if len(isect) > 0:
            bp = int(np.sum(isect[:, 1] - isect[:, 0]))
            if bp > best_bp:
                best_bp = bp
                best_tid = t.id

    # tRNA overlap
    best_trna_bp = 0
    for t in trna_annotation.get_transcripts_in_interval(isl):
        isect = intersect_intervals(isl_arr, t.blocks)
        if len(isect) > 0:
            bp = int(np.sum(isect[:, 1] - isect[:, 0]))
            if bp > best_trna_bp:
                best_trna_bp = bp

    combined_bp = max(best_bp, best_trna_bp)
    rows.append({
        "island_name": isl_name,
        "transcript_id": tx_id,
        "biotype": biotype,
        "island_len": isl_len,
        "best_gencode_bp": best_bp,
        "best_gencode_tid": best_tid,
        "best_trna_bp": best_trna_bp,
        "combined_bp": combined_bp,
        "pct_overlap": 100 * combined_bp / isl_len if isl_len > 0 else 0,
        "has_any_overlap": combined_bp > 0,
        "has_50pct_overlap": (combined_bp / isl_len >= 0.5) if isl_len > 0 else False,
    })

ov = pd.DataFrame(rows)
print(f"\nDone. {len(ov):,} islands processed.")

## Summary statistics

In [ ]:
print("=" * 70)
print("ISLAND \u2194 mm39 ANNOTATION OVERLAP")
print("=" * 70)

n_total = len(ov)
n_any = ov["has_any_overlap"].sum()
n_50 = ov["has_50pct_overlap"].sum()

print(f"\nAll islands (all biotypes):")
print(f"  Total query islands:          {n_total:>8,}")
print(f"  With any overlap:             {n_any:>8,}  ({100*n_any/n_total:.1f}%)")
print(f"  With \u226550% overlap:            {n_50:>8,}  ({100*n_50/n_total:.1f}%)")

print(f"\nBreakdown by source biotype (reference gene):")
print(f"{'biotype':>25s}  {'total':>7s}  {'any_ov':>7s}  {'%':>6s}  {'\u226550%':>7s}  {'%':>6s}")
print("-" * 70)

for bt in ov["biotype"].value_counts().head(15).index:
    sub = ov[ov["biotype"] == bt]
    a = sub["has_any_overlap"].sum()
    h = sub["has_50pct_overlap"].sum()
    print(f"{bt:>25s}  {len(sub):>7,}  {a:>7,}  {100*a/len(sub):>5.1f}%  {h:>7,}  {100*h/len(sub):>5.1f}%")

# lncRNA-specific stats
lnc = ov[ov["biotype"].str.contains("lnc|linc", case=False, na=False)]
print(f"\n--- lncRNA islands specifically ---")
print(f"  Total:         {len(lnc):>8,}")
print(f"  Any overlap:   {lnc['has_any_overlap'].sum():>8,}  ({100*lnc['has_any_overlap'].mean():.1f}%)")
print(f"  \u226550% overlap:  {lnc['has_50pct_overlap'].sum():>8,}  ({100*lnc['has_50pct_overlap'].mean():.1f}%)")

print(f"\n{'=' * 70}")
print("Copy-paste for manuscript:")
print(f"  \u2022 {100*n_any/n_total:.0f}% of projected islands have any overlap with mm39 annotations")
print(f"  \u2022 {100*n_50/n_total:.0f}% have \u226550% exonic overlap")
print(f"  \u2022 lncRNA islands: {100*lnc['has_any_overlap'].mean():.0f}% any overlap, "
      f"{100*lnc['has_50pct_overlap'].mean():.0f}% \u226550%")

## Overlap fraction distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel A: histogram of overlap % (all islands)
ax = axes[0]
ax.hist(ov["pct_overlap"], bins=50, color="#457B9D", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Overlap with mm39 annotation (%)")
ax.set_ylabel("Number of islands")
ax.set_title("A  All projected islands", fontsize=12, fontweight="bold", loc="left")
ax.axvline(50, color="red", ls="--", lw=1, alpha=0.7)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel B: overlap rate by biotype
ax = axes[1]
top_bts = ov["biotype"].value_counts().head(10).index.tolist()
bt_rates = []
for bt in top_bts:
    sub = ov[ov["biotype"] == bt]
    bt_rates.append({
        "biotype": bt,
        "any_overlap": 100 * sub["has_any_overlap"].mean(),
        "over50": 100 * sub["has_50pct_overlap"].mean(),
        "n": len(sub),
    })
bt_df = pd.DataFrame(bt_rates)

y = np.arange(len(bt_df))
ax.barh(y, bt_df["over50"], color="#2A9D8F", edgecolor="white", height=0.6)
for i, row in bt_df.iterrows():
    ax.text(row["over50"] + 1, i, f"n={row['n']:,}", va="center", fontsize=7)
ax.set_yticks(y)
ax.set_yticklabels(bt_df["biotype"], fontsize=9)
ax.set_xlabel("Islands with \u226550% annotation overlap (%)")
ax.set_title("B  By source biotype", fontsize=12, fontweight="bold", loc="left")
ax.set_xlim(0, 109)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "island_annotation_overlap.pdf", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "island_annotation_overlap.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved to {FIGURES_DIR}")